### Experimental custom alerts/recommendation tool

**This is an AI-generated attempt to replace dataprep's data insight -feature.**
 
Double check all significant insights!

**Results should match mostly with dataprep, with small differences!**

In [4]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../houses_fixed.csv")

### Part 1: Data insights

In [5]:
def dataset_insights(df):

    # Imports
    import numpy as np
    import pandas as pd
    from scipy import stats
    import pandas as pd
    import warnings

    # Sets it so it shows all the rows all the time
    pd.set_option("display.max_rows", None)

    # Ignores warnings (for scipy's skewness check)
    warnings.filterwarnings("ignore")

    # We place all our insights in here.
    insights = []

    # For each column, make the following checks...
    for col in df.columns:

        s = df[col]

        # Missing values check (MORE THAN 1% missing values will give an alert)
        missing = s.isna().mean()
        if missing > 0.01:

            # Add it to the insights list
            insights.append({
                "Column": col,
                "Problem Type": "Missing values",
                "Description": f"{missing:.1%} missing values"
            })

        # NUMERICAL COLUMNS 
        # -----------------
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            
            # Drops the missing values (otherwise the checks will break)
            x = s.dropna()

            # If the length of the column in rows is bigger than 7...
            if len(x) > 7:

                # Skewness check (MORE THAN 1 will give an alert)
                skew = stats.skew(x)
                if abs(skew) > 1:
                    direction = "right" if skew > 0 else "left"

                    # Add it to the insights list.
                    insights.append({
                        "Column": col,
                        "Problem Type": "Skewness",
                        "Description": (
                            f"Strongly {direction}-skewed "
                            f"(skewness={skew:.2f})"
                        )
                    })

                # Normality check (if the probability value is less than .1% it will give an alert
                # assuming the null-hypothesis is true, of course.)
                normal = stats.normaltest(x)
                if normal.pvalue < 0.001:

                    # Add it to the insights list.
                    insights.append({
                        "Column": col,
                        "Problem Type": "Normality",
                        "Description": (
                            "Strongly deviates from normality"
                        )
                    })

            # Checking for zeros (MORE THAN 5% = alert)
            zero_pct = (x == 0).mean()
            if zero_pct > 0.05:

                # Add it to the insights list.
                insights.append({
                    "Column": col,
                    "Problem Type": "Zeros",
                    "Description": f"{zero_pct:.1%} zero values"
                })

            # Checking for negative values (MORE THAN 1% = alert)
            negative_pct = (x < 0).mean()

            if negative_pct > 0.01:

                # Add it to the insights list.
                insights.append({
                    "Column": col,
                    "Problem Type": "Negative values",
                    "Description": f"{negative_pct:.1%} negative values"
                })

            # Checking for outliers (IF THERE ARE ANY = alert. Even 1.)
            q1, q3 = x.quantile([.25, .75])
            iqr = q3 - q1

            outliers = (
                (x < q1 - 1.5 * iqr) |
                (x > q3 + 1.5 * iqr)
            ).sum()

            # If there are outliers, add it to the insights list.
            if outliers:
                insights.append({
                    "Column": col,
                    "Problem Type": "Outliers",
                    "Description": f"{outliers:,} IQR outliers"
                })

        # CATEGORICAL COLUMNS
        else:
            # Uniqueness (If all the values are unique and it's a categorical column = Alert)
            nunique = s.nunique(dropna=True)
            if nunique == len(s.dropna()):
                insights.append({
                    "Column": col,
                    "Problem Type": "Uniqueness",
                    "Description": "All non-null values in this categorical column are unique."
                })

            # High-Cardinality (If more than 50 values are unique = Alert)
            if nunique > 50:
                insights.append({
                    "Column": col,
                    "Problem Type": "High cardinality",
                    "Description": f"{nunique:,} unique values"
                })

    # Sort by Problem Type
    insights = pd.DataFrame(insights).sort_values("Problem Type", ascending=False)

    return insights

dataset_insights(df)

,Column,Problem Type,Description
23,view,Zeros,90.2% zero values
40,yr_renovated,Zeros,95.8% zero values
19,waterfront,Zeros,99.2% zero values
35,sqft_basement,Zeros,60.7% zero values
21,view,Skewness,Strongly right-skewed (skewness=3.40)
30,sqft_above,Skewness,Strongly right-skewed (skewness=1.45)
17,waterfront,Skewness,Strongly right-skewed (skewness=11.38)
33,sqft_basement,Skewness,Strongly right-skewed (skewness=1.58)
13,sqft_lot,Skewness,Strongly right-skewed (skewness=13.06)
25,condition,Skewness,Strongly right-skewed (skewness=1.03)


### Part 2: Extremely similar distributions

In [7]:
import numpy as np
import pandas as pd

from itertools import combinations
from scipy.stats import ks_2samp, wasserstein_distance


def similar_distributions(
    df,
    columns=None,
    threshold=0.2,
    min_samples=20,
    numeric_only=True,
):
    """
    Find pairs of variables with similar empirical distributions.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.

    columns : list[str], optional
        Columns to compare. If None, all suitable columns are used.

    threshold : float, default=0.2
        Maximum combined similarity score to report.
        Lower = more similar.

    min_samples : int, default=20
        Minimum number of non-null observations required per variable.

    numeric_only : bool, default=True
        If True, only numeric columns are considered.

    Returns
    -------
    pd.DataFrame
        Ranked pairs of variables with similarity metrics.
    """

    if columns is None:
        if numeric_only:
            columns = df.select_dtypes(include="number").columns.tolist()
        else:
            columns = df.columns.tolist()

    # Remove unsuitable columns
    data = {}

    for col in columns:
        x = pd.to_numeric(df[col], errors="coerce").dropna()

        if len(x) >= min_samples and x.nunique() > 1:
            data[col] = x.to_numpy()

    results = []

    for col1, col2 in combinations(data, 2):
        x = data[col1]
        y = data[col2]

        # KS test
        ks_stat, ks_p = ks_2samp(x, y)

        # Wasserstein distance
        wd = wasserstein_distance(x, y)

        # Normalize Wasserstein by pooled standard deviation.
        # This makes the value somewhat comparable across variables
        # with different scales.
        pooled_std = np.std(np.concatenate([x, y]))

        if pooled_std > 0:
            wd_normalized = wd / pooled_std
        else:
            wd_normalized = 0.0

        # Combined score.
        #
        # KS is already in [0, 1].
        # Normalize Wasserstein using 1 - exp(-distance), which
        # keeps it in [0, 1] without imposing a hard cutoff.
        wd_score = 1 - np.exp(-wd_normalized)

        score = 0.5 * ks_stat + 0.5 * wd_score

        if score <= threshold:
            results.append({
                "variable_1": col1,
                "variable_2": col2,
                "similarity_score": score,
                "ks_statistic": ks_stat,
                "ks_pvalue": ks_p,
                "wasserstein": wd,
                "wasserstein_normalized": wd_normalized,
            })

    if not results:
        return pd.DataFrame(
            columns=[
                "variable_1",
                "variable_2",
                "similarity_score",
                "ks_statistic",
                "ks_pvalue",
                "wasserstein",
                "wasserstein_normalized",
            ]
        )

    df = pd.DataFrame(results).sort_values("similarity_score").reset_index(drop=True)

    df["similar_distributions"] = "Yes"

    columns = df.columns.tolist()
    columns.insert(2, columns.pop(columns.index("similar_distributions")))
    df = df[columns]

    df = df[df["similarity_score"] < .1]

    numeric_cols = df.select_dtypes(include="number").columns
    df[numeric_cols] = df[numeric_cols].round(3)

    return df

similar_distributions(df)

,variable_1,variable_2,similar_distributions,similarity_score,ks_statistic,ks_pvalue,wasserstein,wasserstein_normalized
0,sqft_lot,sqft_lot15,Yes,0.051,0.035,0.0,2437.99,0.069
